In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm import tqdm

## Class Dataset with Fixes

### Bound the poses, extraction of camera intrinsics before coordinates manipulation, convert LLFF format to OpenGL format, normalize poses by centering at origin and scale to fit the sphere

In [ ]:
class MultiResNeRFDataset(torch.utils.data.Dataset):
    def __init__(self, image_dirs, poses_path, mode="train", device="cpu"):
        
        super().__init__()
        self.device = device
        self.mode = mode
        
        
        raw_poses_bounds = np.load(poses_path)
        poses = raw_poses_bounds[:, :-2].reshape([-1, 3, 5])
        bds = raw_poses_bounds[:, -2:]
        
        
        self.base_hwf = poses[0, :, 4].copy() 
        
        ## LLFF to OpenGL coordinate system conversion
        poses_corrected = np.concatenate([
            poses[:, 1:2, :],   # New X: Right
            -poses[:, 0:1, :],  # New Y: Up (-Down)
            poses[:, 2:3, :]    # New Z: Backward
        ], axis=1)
        
        poses_3x4 = poses_corrected[:, :3, :4]
        
        #Center and scale poses
        centers = poses_3x4[:, :3, 3]
        center_mean = np.mean(centers, axis=0)
        poses_3x4[:, :3, 3] -= center_mean # Center trajectory
        
        max_dist = np.max(np.linalg.norm(poses_3x4[:, :3, 3], axis=1))
        scale_factor = 1.0 / (max_dist + 1e-5)
        poses_3x4[:, :3, 3] *= scale_factor # Fit inside [-1, 1] bounds
        bds *= scale_factor
        
        self.poses = torch.from_numpy(poses_3x4).float()
        self.bounds = torch.from_numpy(bds).float()
        
        # 2. Gather Image Paths and Map to Pose Indices
        self.samples = []
        for img_dir in image_dirs:
            # Determine downsample factor from folder name (e.g., images_4 -> 4)
            folder_name = os.path.basename(img_dir)
            if folder_name == "images":
                factor = 1.0
            elif folder_name.startswith("images_"):
                factor = float(folder_name.split("_")[1])
            else:
                factor = 1.0
                
            files = sorted(os.listdir(img_dir))
            for i, fname in enumerate(files):
                # Ensure we match the image to the correct pose index (0 to 193)
                self.samples.append({
                    "path": os.path.join(img_dir, fname),
                    "pose_idx": i,
                    "factor": factor
                })
                
        print(f"[{mode.upper()}] Loaded {len(self.samples)} images from {len(image_dirs)} directories.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        pose_idx = sample["pose_idx"]
        factor = sample["factor"]
        
        # Calculate dynamic intrinsics for this specific resolution
        H = int(self.base_hwf[0] // factor)
        W = int(self.base_hwf[1] // factor)
        focal = self.base_hwf[2] / factor
        
        # Load Image
        img = Image.open(sample["path"]).convert("RGB")
        # Resize dynamically if needed to match exact HxW integer rounding
        if img.size != (W, H):
            img = img.resize((W, H), Image.Resampling.LANCZOS)
        
        img_tensor = torch.from_numpy(np.array(img)).float() / 255.0
        
        return {
            "image": img_tensor.to(self.device),
            "pose": self.poses[pose_idx].to(self.device),
            "bounds": self.bounds[pose_idx].to(self.device),
            "H": H, "W": W, "focal": focal,
            "filename": os.path.basename(sample["path"])
        }

## Finite sphere function

In [ ]:
def apply_mipnerf360_contraction(x):
    """Warps infinite space into a finite sphere."""
    norm = torch.norm(x, dim=-1, keepdim=True)
    return torch.where(norm <= 1.0, x, (2.0 - 1.0 / norm) * (x / norm))

## Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, L_pos=10, L_dir=4):
        super().__init__()
        self.freqs_pos = 2.0 ** torch.linspace(0, L_pos - 1, L_pos)
        self.freqs_dir = 2.0 ** torch.linspace(0, L_dir - 1, L_dir)

    def forward(self, x, freqs):
        out = [x]
        for freq in freqs.to(x.device):
            out.append(torch.sin(x * freq))
            out.append(torch.cos(x * freq))
        return torch.cat(out, dim=-1)

## NeRF model Simple

In [ ]:
class NeRFMLP(nn.Module):
    def __init__(self, L_pos=10, L_dir=4, hidden_dim=256):
        super().__init__()
        self.encoder = PositionalEncoding(L_pos, L_dir)
        in_pos = 3 + (3 * 2 * L_pos)
        in_dir = 3 + (3 * 2 * L_dir)
        
        self.layer1 = nn.Linear(in_pos, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.layer4 = nn.Linear(hidden_dim, hidden_dim)
        self.layer5 = nn.Linear(hidden_dim + in_pos, hidden_dim)
        self.layer6 = nn.Linear(hidden_dim, hidden_dim)
        self.layer7 = nn.Linear(hidden_dim, hidden_dim)
        self.layer8 = nn.Linear(hidden_dim, hidden_dim)
        
        self.sigma_out = nn.Linear(hidden_dim, 1)
        self.feature_out = nn.Linear(hidden_dim, hidden_dim)
        self.rgb_layer1 = nn.Linear(hidden_dim + in_dir, hidden_dim // 2)
        self.rgb_out = nn.Linear(hidden_dim // 2, 3)

    def forward(self, points, dirs):
        contracted_points = apply_mipnerf360_contraction(points)
        pos_enc = self.encoder(contracted_points, self.encoder.freqs_pos)
        dir_enc = self.encoder(dirs, self.encoder.freqs_dir)
        
        h = F.relu(self.layer1(pos_enc))
        h = F.relu(self.layer2(h))
        h = F.relu(self.layer3(h))
        h = F.relu(self.layer4(h))
        h = F.relu(self.layer5(torch.cat([h, pos_enc], dim=-1)))
        h = F.relu(self.layer6(h))
        h = F.relu(self.layer7(h))
        h = self.layer8(h)
        
        sigma = F.softplus(self.sigma_out(h))
        geo_feature = self.feature_out(h)
        
        h_color = torch.cat([geo_feature, dir_enc], dim=-1)
        h_color = F.relu(self.rgb_layer1(h_color))
        rgb = torch.sigmoid(self.rgb_out(h_color))
        
        return rgb, sigma

## Ray Marching and rendering

In [ ]:
def get_rays(H, W, focal, pose):
    """Generates Ray origins and directions."""
    i, j = torch.meshgrid(torch.linspace(0, W-1, W), torch.linspace(0, H-1, H), indexing='xy')
    dirs = torch.stack([(i - W * 0.5) / focal, -(j - H * 0.5) / focal, -torch.ones_like(i)], -1).to(pose.device)
    rays_d = torch.sum(dirs[..., None, :] * pose[:3, :3], -1) 
    rays_o = pose[:3, 3].expand(rays_d.shape)
    return rays_o, rays_d

def render_rays(model, rays_o, rays_d, bounds, num_samples=64):
    near, far = bounds[0], bounds[1]
    t_vals = torch.linspace(0., 1., steps=num_samples).to(rays_o.device)
    z_vals = near + (far - near) * t_vals
    
    if model.training:
        mids = .5 * (z_vals[..., 1:] + z_vals[..., :-1])
        upper = torch.cat([mids, z_vals[..., -1:]], -1)
        lower = torch.cat([z_vals[..., :1], mids], -1)
        t_rand = torch.rand(z_vals.shape).to(rays_o.device)
        z_vals = lower + (upper - lower) * t_rand

    pts = rays_o[..., None, :] + rays_d[..., None, :] * z_vals[..., :, None]
    dirs = rays_d[..., None, :].expand(pts.shape)
    
    pts_flat, dirs_flat = pts.reshape(-1, 3), dirs.reshape(-1, 3)
    rgb_flat, sigma_flat = model(pts_flat, dirs_flat)
    
    rgb, sigma = rgb_flat.reshape(pts.shape[:2] + (3,)), sigma_flat.reshape(pts.shape[:2])
    
    dists = z_vals[..., 1:] - z_vals[..., :-1]
    dists = torch.cat([dists, torch.tensor([1e10]).expand(dists[..., :1].shape).to(rays_o.device)], -1)
    
    alpha = 1.0 - torch.exp(-sigma * dists)
    transmittance = torch.cumprod(torch.cat([torch.ones((alpha.shape[0], 1)).to(rays_o.device), 1. - alpha + 1e-10], -1), -1)[:, :-1]
    weights = alpha * transmittance
    
    rgb_map = torch.sum(weights[..., None] * rgb, -2)
    return rgb_map, sigma

## Train and Testing Loops

In [ ]:
def train_and_test():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using Device: {device}")
    
    # Kaggle Paths
    base_dir = "/kaggle/input/datasets/thnhdg/testing/360_v2/bicycle"
    poses_path = os.path.join(base_dir, "poses_bounds.npy")
    
    train_dirs = [
        os.path.join(base_dir, "images"),
        os.path.join(base_dir, "images_2"),
        os.path.join(base_dir, "images_4")
    ]
    test_dirs = [os.path.join(base_dir, "images_8")]
    
    # Initialize Datasets (Tensors on CPU to prevent VRAM overflow from high-res images)
    train_dataset = MultiResNeRFDataset(train_dirs, poses_path, mode="train", device="cpu")
    test_dataset = MultiResNeRFDataset(test_dirs, poses_path, mode="test", device="cpu")
    
    model = NeRFMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    
    EPOCHS = 10
    BATCH_RAYS = 2048
    LAMBDA_TV = 1e-4
    best_psnr = 0.0
    
    # --- TRAINING LOOP ---
    for epoch in range(EPOCHS):
        model.train()
        total_loss, total_psnr = 0, 0
        
        # Shuffle train data manually for diversity
        indices = torch.randperm(len(train_dataset)).tolist()
        
        for idx in tqdm(indices, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
            data = train_dataset[idx]
            target_img = data["image"].to(device)
            pose = data["pose"].to(device)
            bounds = data["bounds"].to(device)
            
            rays_o, rays_d = get_rays(data["H"], data["W"], data["focal"], pose)
            rays_o_flat, rays_d_flat = rays_o.reshape(-1, 3), rays_d.reshape(-1, 3)
            target_flat = target_img.reshape(-1, 3)
            
            # Sub-sample rays for this step
            select_inds = torch.randperm(rays_o_flat.shape[0])[:BATCH_RAYS]
            pred_rgb, pred_sigma = render_rays(model, rays_o_flat[select_inds], rays_d_flat[select_inds], bounds)
            
            loss_color = F.mse_loss(pred_rgb, target_flat[select_inds])
            loss_tv = torch.mean(torch.abs(pred_sigma[:, 1:] - pred_sigma[:, :-1]))
            loss = loss_color + (LAMBDA_TV * loss_tv)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            total_psnr += (-10.0 * torch.log10(loss_color)).item()
            
        avg_psnr = total_psnr / len(train_dataset)
        print(f"Epoch {epoch+1} Complete | Avg Train PSNR: {avg_psnr:.2f} dB")
        
        if avg_psnr > best_psnr:
            best_psnr = avg_psnr
            torch.save(model.state_dict(), 'best_nerf_weights.pth')
            
    # --- TESTING LOOP ---
    print("\nStarting Inference on Test Set (images_8)...")
    model.load_state_dict(torch.load('best_nerf_weights.pth'))
    model.eval()
    
    output_dir = "/kaggle/working/render_output"
    os.makedirs(output_dir, exist_ok=True)
    test_psnr_accum = 0.0
    CHUNK_SIZE = 4096 
    
    with torch.no_grad():
        for i in tqdm(range(len(test_dataset)), desc="Testing"):
            data = test_dataset[i]
            target_img = data["image"].to(device)
            pose = data["pose"].to(device)
            bounds = data["bounds"].to(device)
            
            rays_o, rays_d = get_rays(data["H"], data["W"], data["focal"], pose)
            rays_o_flat, rays_d_flat = rays_o.reshape(-1, 3), rays_d.reshape(-1, 3)
            
            rendered_pixels = []
            for start in range(0, rays_o_flat.shape[0], CHUNK_SIZE):
                end = start + CHUNK_SIZE
                rgb_chunk, _ = render_rays(model, rays_o_flat[start:end], rays_d_flat[start:end], bounds)
                rendered_pixels.append(rgb_chunk)
                
            full_rgb = torch.cat(rendered_pixels, dim=0).reshape(data["H"], data["W"], 3).clamp(0, 1)
            
            img_mse = F.mse_loss(full_rgb, target_img)
            test_psnr_accum += (-10.0 * torch.log10(img_mse)).item()
            
            rendered_np = (full_rgb.cpu().numpy() * 255.0).astype(np.uint8)
            Image.fromarray(rendered_np).save(os.path.join(output_dir, f"render_{data['filename']}"))
            
    print(f"\nFinal Test Set PSNR: {test_psnr_accum / len(test_dataset):.2f} dB")
    print(f"Outputs saved to {output_dir}")

## Main code

In [ ]:
if __name__ == "__main__":
    train_and_test()